# S4_5 — Masking Strategy Ablation Study
**Research purpose:** Validates the disease-region-biased masking as an independent contribution.
Trains a second Leaf-JEPA model with identical hyperparameters but STANDARD (uniform) masking.
The difference in linear probe F1 quantifies the masking contribution.

**Compute note:** This runs a full second pretraining — plan accordingly.
If compute is limited, reduce to 50 epochs for both models and compare.

RQ contribution: "Biased masking adds X pp to linear probe macro-F1 over standard masking."


In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import sys, json, copy
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import torch
import timm
import wandb

from stage4_leaf_jepa_pretraining.config_stage4 import *
from stage4_leaf_jepa_pretraining.pretrain_utils import (
    set_seed, get_pretrain_transform, PlantVillagePretrainDataset,
    MultiBlockMasking, IJEPAPredictor, EMAUpdater, get_layerwise_optimizer,
    WarmupCosineScheduler, pretrain_one_epoch, LinearProbeMonitor
)

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ablation uses fewer epochs for compute efficiency
ABLATION_EPOCHS = min(PT_EPOCHS, 50)
print(f"Masking ablation: {ABLATION_EPOCHS} epochs per variant")


In [ ]:
# ── Cell 2: Check if biased model results already exist ───────────────────────
biased_lp_path   = PRETRAIN_DIR / "lp_monitor_history.json"
standard_lp_path = PRETRAIN_DIR / "ablation_standard_lp_history.json"

biased_lp_history = []
if biased_lp_path.exists():
    with open(biased_lp_path) as f:
        biased_lp_history = json.load(f)
    # Filter to ablation epoch range
    biased_lp_history = [r for r in biased_lp_history if r["pretrain_epoch"] <= ABLATION_EPOCHS]
    print(f"Biased masking LP history loaded: {len(biased_lp_history)} entries")
    if biased_lp_history:
        best = max(biased_lp_history, key=lambda r: r["lp_val_macro_f1"])
        print(f"  Best biased LP F1 @ epoch ≤{ABLATION_EPOCHS}: {best['lp_val_macro_f1']:.4f}")
else:
    print("⚠️  Biased masking results not found. Run S4_3 first.")


In [ ]:
# ── Cell 3: Train standard masking baseline ───────────────────────────────────
def build_fresh_models():
    enc = timm.create_model(VIT_MODEL_NAME, pretrained=False, num_classes=0, global_pool="")
    ckpt = torch.load(IJEPA_CHECKPOINT, map_location="cpu")
    sd   = ckpt.get("target_encoder", ckpt.get("encoder", ckpt))
    sd   = {k.replace("module.", ""): v for k, v in sd.items()}
    enc.load_state_dict(sd, strict=False)
    tgt = copy.deepcopy(enc)
    for p in tgt.parameters(): p.requires_grad = False
    pred = IJEPAPredictor(
        encoder_embed_dim=VIT_EMBED_DIM, pred_embed_dim=PRED_EMBED_DIM,
        num_patches=NUM_PATCHES, num_heads=PRED_NUM_HEADS, depth=PRED_DEPTH,
    )
    return enc.to(device), tgt.to(device), pred.to(device)

if not standard_lp_path.exists():
    print("Training standard masking baseline for ablation...")
    set_seed(RANDOM_SEED)

    ctx_std, tgt_std, pred_std = build_fresh_models()
    masking_std = MultiBlockMasking(
        image_size=IMAGE_CROP, patch_size=PATCH_SIZE,
        num_target_blocks=PT_NUM_TARGET_BLOCKS,
        context_scale=PT_CONTEXT_SCALE, context_ratio=PT_CONTEXT_RATIO,
        target_scale=PT_TARGET_SCALE, target_ratio=PT_TARGET_RATIO,
    )

    transform = get_pretrain_transform(NORM_MEAN, NORM_STD, IMAGE_CROP, IMAGE_RESIZE)
    dataset   = PlantVillagePretrainDataset(SPLITS_DIR / "plantvillage_splits.csv", transform)
    loader    = torch.utils.data.DataLoader(
        dataset, batch_size=PT_BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True,
    )

    opt_std = get_layerwise_optimizer(
        ctx_std, pred_std, FROZEN_LAYERS, LOW_LR_LAYERS, STD_LR_LAYERS,
        PT_LR_HEAD, PT_LR_ENCODER_MID, PT_LR_ENCODER_TOP, PT_WEIGHT_DECAY,
    )
    sched_std = WarmupCosineScheduler(opt_std, PT_WARMUP_EPOCHS, ABLATION_EPOCHS)
    ema_std   = EMAUpdater(EMA_TAU_START, EMA_TAU_END,
                            total_steps=ABLATION_EPOCHS * len(loader))

    lp_std = LinearProbeMonitor(SPLITS_DIR, NORM_MEAN, NORM_STD, NUM_CLASSES,
                                  monitor_epochs=LP_MONITOR_EPOCHS,
                                  monitor_frac=LP_MONITOR_FRAC,
                                  device=device)

    run_std = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                          name=f"LeafJEPA-pretrain-standard-ablation-{ABLATION_EPOCHS}e",
                          config={"masking": "standard", "epochs": ABLATION_EPOCHS},
                          reinit=True)

    std_history = []
    for epoch in range(1, ABLATION_EPOCHS + 1):
        sched_std.step(epoch - 1)
        metrics = pretrain_one_epoch(
            ctx_std, tgt_std, pred_std, loader, masking_std, None,
            opt_std, ema_std, device, epoch, USE_AMP, PT_ACCUMULATE_GRAD,
            PT_LOSS, run_std,
        )
        std_history.append(metrics)
        print(f"  Epoch {epoch}/{ABLATION_EPOCHS} | Loss: {metrics['loss']:.4f}")

        if epoch % LP_MONITOR_INTERVAL == 0 or epoch == ABLATION_EPOCHS:
            lp_std.run(tgt_std, epoch, run_std)

    if run_std: run_std.finish()
    with open(standard_lp_path, "w") as f:
        json.dump(lp_std.history, f, indent=2)
    print("✅ Standard masking ablation complete")
else:
    with open(standard_lp_path) as f:
        std_lp_data = json.load(f)
    print(f"Standard masking LP history loaded: {len(std_lp_data)} entries")


In [ ]:
# ── Cell 4: Ablation comparison ───────────────────────────────────────────────
import matplotlib.pyplot as plt

with open(standard_lp_path) as f:
    std_lp_data = json.load(f)

std_epochs = [r["pretrain_epoch"]   for r in std_lp_data]
std_f1s    = [r["lp_val_macro_f1"]  for r in std_lp_data]
bia_epochs = [r["pretrain_epoch"]   for r in biased_lp_history] if biased_lp_history else []
bia_f1s    = [r["lp_val_macro_f1"]  for r in biased_lp_history] if biased_lp_history else []

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(std_epochs, std_f1s, color="#90caf9", lw=2, marker="o", ms=6, label="Standard masking (ablation)")
if bia_f1s:
    ax.plot(bia_epochs, bia_f1s, color="#f59e0b", lw=2, marker="s", ms=6, label="Biased masking (novel, ours)")

if std_f1s and bia_f1s:
    best_std = max(std_f1s)
    best_bia = max(bia_f1s)
    delta    = best_bia - best_std
    ax.annotate(f"Δ = {delta:+.4f}\n({delta*100:.1f} pp)",
                xy=(bia_epochs[bia_f1s.index(best_bia)], best_bia),
                xytext=(bia_epochs[bia_f1s.index(best_bia)] - 8, best_bia - 0.03),
                arrowprops=dict(arrowstyle="->", color="#ef5350"),
                fontsize=11, color="#ef5350")
    print(f"\nAblation Result:")
    print(f"  Standard masking:  {best_std:.4f}")
    print(f"  Biased masking:    {best_bia:.4f}")
    print(f"  Delta:             {delta:+.4f} ({delta*100:.1f} pp)")
    print(f"\n  Dissertation: 'Disease-region-biased masking improves linear probe")
    print(f"   macro-F1 by {delta*100:.1f} percentage points over standard I-JEPA masking,")
    print(f"   validating the contribution of the novel target block sampling strategy.'")

ax.set_xlabel("Pretrain Epoch"); ax.set_ylabel("Val Macro-F1 (Linear Probe)")
ax.set_title("Masking Strategy Ablation\nStandard vs Disease-Region-Biased Masking", fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "S4_masking_ablation.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Save results
ablation_results = {
    "standard_best_lp_f1": max(std_f1s) if std_f1s else None,
    "biased_best_lp_f1":   max(bia_f1s) if bia_f1s else None,
    "delta":               (max(bia_f1s) - max(std_f1s)) if (bia_f1s and std_f1s) else None,
    "ablation_epochs":     ABLATION_EPOCHS,
}
with open(PRETRAIN_DIR / "S4_masking_ablation.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
print("\n✅ Ablation figure and results saved.")
